# Compiling Train/Test Sets
---
Begin by sorting synthetic arrays by RC (prove this is a good metric to assess array)

Construct input data vectors for training (geometry/frequency band) and target variable (RC) for both seismic and infrasound arrays

Training data can be obtained upon request (mnronac@sandia.gov)

In [1]:
%%time
import warnings, sys, os, glob, random, pickle
warnings.filterwarnings("ignore")
#-----------------------------------------------------------------------------------------------------------------#
# Import Cardinal
pwd = os.getcwd(); current_subdir = os.path.dirname(pwd)
sys.path.append(current_subdir); import cardinal
#-----------------------------------------------------------------------------------------------------------------#
# Import packages as
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#-----------------------------------------------------------------------------------------------------------------#
# From packages import functions
from sklearn.model_selection import KFold

CPU times: user 7.13 s, sys: 4.62 s, total: 11.8 s
Wall time: 30.5 s


## Construct Frequency Bands
---

In [2]:
f_bands = cardinal.make_custom_fbands(f_min=0.02, f_max=21, type='octave')
f_bands

,band,fmin,fcenter,fmax,win,step
0,1.0,0.02,0.03,0.04,134.270734,13.427073
1,2.0,0.04,0.06,0.08,68.541468,6.854147
2,3.0,0.08,0.12,0.16,35.676835,3.567684
3,4.0,0.16,0.24,0.32,19.244519,1.924452
4,5.0,0.32,0.48,0.64,11.028360,1.102836
5,6.0,0.64,0.96,1.28,6.920281,0.692028
6,7.0,1.28,1.92,2.56,4.866242,0.486624
7,8.0,2.56,3.84,5.12,3.839222,0.383922
8,9.0,5.12,7.68,10.24,3.325712,0.332571
9,10.0,10.24,15.36,20.48,3.068957,0.306896


## Sort Synthetic Arrays by RC
---

In [188]:
%%time
band_idxs = [1, 5, 9] # just do bands 2, 6, and 10
for band_idx in band_idxs:
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, std)
    directory_path = 'Synthetic_Array_Revised_Max/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain and spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, std)
    RC = []; subarray = []
    for max_file, std_file in zip(RC_max_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_std_i = np.load(std_file); RC.append(RC_std_i / RC_max_i)
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 2
Worst configuration: Synthetic_Array_Revised_Max/Band_2/Plots/5740_S2_SC_SB_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_2/Plots/40356_SE_S1_S4_S5_S2_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_2/Plots/14575_S1_S4_S5_config.png
Start band 6
Worst configuration: Synthetic_Array_Revised_Max/Band_6/Plots/52019_S5_SE_S3_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_6/Plots/26361_SE_S1_SA_SC_S5_S4_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_6/Plots/7349_S2_SA_SD_S3_SE_SB_SC_config.png
Start band 10
Worst configuration: Synthetic_Array_Revised_Max/Band_10/Plots/202020_S5_SD_S1_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_10/Plots/18834_S1_S3_SA_S2_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_10/Plots/102407_SE_S5_S1_S3_SA_SC_SB_config.png
CPU times: user 53.3 s, sys: 30.2 s, total: 1min 23s
Wall time: 17min 24s


In [189]:
%%time
band_idxs = [8] # just doing band 9
for band_idx in band_idxs:
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, std)
    directory_path = 'Synthetic_Array_Revised_Max/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain and spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, std)
    RC = []; subarray = []
    for max_file, std_file in zip(RC_max_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_std_i = np.load(std_file); RC.append(RC_std_i / RC_max_i)
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 9
Worst configuration: Synthetic_Array_Revised_Max/Band_9/Plots/124080_S3_SB_S4_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_9/Plots/186565_S1_S3_SB_S2_SC_SE_SD_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_9/Plots/242252_SE_SB_SD_S5_S4_SA_SC_config.png
CPU times: user 16.8 s, sys: 9.87 s, total: 26.7 s
Wall time: 4min 25s


In [3]:
%%time
band_idxs = [2] # just doing band 3
for band_idx in band_idxs:
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, std)
    directory_path = 'Synthetic_Array_Revised_Max/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain and spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, std)
    RC = []; subarray = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_mean_i = np.load(mean_file); RC_std_i = np.load(std_file)
        if RC_max_i < 2000:
            continue
        RC.append( (RC_mean_i + RC_std_i) / RC_max_i) # combine into a single metric
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 3
Worst configuration: Synthetic_Array_Revised_Max/Band_3/Plots/15998_SC_S5_S1_S4_S2_S3_SD_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_3/Plots/24217_S4_S5_SE_S2_S1_SC_SB_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_3/Plots/212867_S3_SD_SE_SB_SC_SA_S4_config.png
CPU times: user 55.3 s, sys: 45.5 s, total: 1min 40s
Wall time: 7min 44s


## Construct Databases - Infrasound
---
Subarray geometry, frequency range, and RC

In [4]:
%%time
X_geos = []; X_freqrange = []; y_RC = []
for band_idx in range(len(f_bands)):
    # Get filepaths for subarray geometries, freq ranges, and RC (max, mean, std)
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    geo_wildcard_pattern = 'Data/*relative_geometry.npy' # geometries already in meters
    freqmin_wildcard_pattern = 'Data/*freqmin.npy'; freqmax_wildcard_pattern = 'Data/*freqmax.npy'
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain, average energy, spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    geo_files = sorted(glob.glob(directory_path + '/' + geo_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmin_files = sorted(glob.glob(directory_path + '/' + freqmin_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmax_files = sorted(glob.glob(directory_path + '/' + freqmax_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get relative geometries and frequency ranges
    geos = []; freqrange = []
    for geo_file, freqmin_file, freqmax_file in zip(geo_files, freqmin_files, freqmax_files):
        geo_i = np.load(geo_file); freqmin_i = np.load(freqmin_file); freqmax_i = np.load(freqmax_file)
        ref_idx = np.where((geo_i == 0.))
        try:
            ref_idx = np.where((geo_i == 0.)); geo_i[ref_idx] = np.array([1e-10, 1e-10]) # need to mask out zeros for standardization
        except:
            for zero_idx in range(len(ref_idx[0])):
                geo_i[ref_idx[0][zero_idx], ref_idx[1][zero_idx]] = 1e-10
        geos.append(geo_i); freqrange.append(np.array([freqmin_i, freqmax_i]))
    X_geos.append(geos); X_freqrange.append(freqrange)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC_max = []; RC_mean = []; RC_std = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_max.append(RC_max_i)
        RC_mean_i = np.load(mean_file); RC_mean.append(RC_mean_i)
        RC_std_i = np.load(std_file); RC_std.append(RC_std_i)
    RC_max = np.array(RC_max).reshape(-1,1); RC_mean = np.array(RC_mean).reshape(-1,1); RC_std = np.array(RC_std).reshape(-1,1)
    RC = np.zeros((RC_max.shape[0],3))
    RC[:,0] = RC_max[:,0]; RC[:,1] = RC_mean[:,0]; RC[:,2] = RC_std[:,0] # 1st column is max, 2nd column is mean, 3rd column is std
    y_RC.append(RC)
    print('Done with band: '+ str(band_idx+1))

Done with band: 1
Done with band: 2
Done with band: 3
Done with band: 4
Done with band: 5
Done with band: 6
Done with band: 7
Done with band: 8
Done with band: 9
Done with band: 10
CPU times: user 22min 8s, sys: 15min 19s, total: 37min 27s
Wall time: 3h 15min 48s


## Construct Databases - Seismic
---
Subarray geometry, frequency range, and RC

In [10]:
# Need to check to make sure that each synthetic array has all parameters (have to do this because generating synthetic arrays unexpectedly stopped due to no space in hard drive)
for band_idx in range(len(f_bands)):
    print('Start band '+str(band_idx+1))
    # Get filepaths for subarray geometries, freq ranges, and RC (max, mean, std)
    directory_path = '/Volumes/Extreme SSD/RC_ConvFormer/Synthetic_Arrays/Band_'+str(band_idx+1)
    geo_wildcard_pattern = 'Data/*relative_geometry.npy' # geometries already in meters
    freqmin_wildcard_pattern = 'Data/*freqmin.npy'; freqmax_wildcard_pattern = 'Data/*freqmax.npy'
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain, average energy, spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    print('Sorting files')
    geo_files = sorted(glob.glob(directory_path + '/' + geo_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmin_files = sorted(glob.glob(directory_path + '/' + freqmin_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmax_files = sorted(glob.glob(directory_path + '/' + freqmax_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))  
    #-----------------------------------------------------------------------------------------------------------------#
    # Print length of each file list
    print('Length of relative geometry ' + str(len(geo_files)))
    print('Length of freqmin '+ str(len(freqmin_files)))
    print('Length of freqmax '+ str(len(freqmax_files)))
    print('Length of RC max '+ str(len(RC_max_files)))
    print('Length of RC mean '+ str(len(RC_mean_files)))
    print('Length of RC std '+ str(len(RC_std_files)))
    # Need to check if some params are missing for some synthetic arrays
    revised_geo_files = []
    for geo_file in geo_files:
        revised_geo_files.append(geo_file[:-13]+'.npy') # remove geometry suffix from relative_geometry

    # List of all parameter file lists
    file_lists = [revised_geo_files, freqmin_files, freqmax_files, RC_max_files, RC_mean_files, RC_std_files]

    # Extract core filenames (remove suffix)
    core_filename_sets = []
    for file_list in file_lists:
        core_filenames = {fname.rsplit("_", 1)[0] for fname in file_list}  # Remove last underscore & suffix
        core_filename_sets.append(core_filenames)

    # Find core filenames present in all lists (intersection)
    common_filenames = set.intersection(*core_filename_sets)

    # Find core filenames that are missing from at least one list (union - intersection)
    all_filenames = set.union(*core_filename_sets)
    missing_filenames = all_filenames - common_filenames

    print("Core filenames that are missing parameters:", missing_filenames)

Start band 1
Sorting files
Length of relative geometry 11040
Length of freqmin 11040
Length of freqmax 11040
Length of RC max 11040
Length of RC mean 11040
Length of RC std 11040
Core filenames that are missing parameters: set()
Start band 2
Sorting files
Length of relative geometry 10926
Length of freqmin 10926
Length of freqmax 10926
Length of RC max 10926
Length of RC mean 10926
Length of RC std 10926
Core filenames that are missing parameters: set()
Start band 3
Sorting files
Length of relative geometry 10900
Length of freqmin 10900
Length of freqmax 10900
Length of RC max 10900
Length of RC mean 10900
Length of RC std 10900
Core filenames that are missing parameters: set()
Start band 4
Sorting files
Length of relative geometry 10910
Length of freqmin 10910
Length of freqmax 10910
Length of RC max 10910
Length of RC mean 10910
Length of RC std 10910
Core filenames that are missing parameters: set()
Start band 5
Sorting files
Length of relative geometry 11199
Length of freqmin 11199

In [43]:
%%time
X_geos_seismic = []; X_freqrange_seismic = []; y_RC_seismic = []
for band_idx in range(len(f_bands)):
    # Get filepaths for subarray geometries, freq ranges, and RC (max, mean, std)
    directory_path = '/Volumes/Extreme SSD/RC_ConvFormer/Synthetic_Arrays/Band_'+str(band_idx+1)
    geo_wildcard_pattern = 'Data/*relative_geometry.npy' # geometries already in meters
    freqmin_wildcard_pattern = 'Data/*freqmin.npy'; freqmax_wildcard_pattern = 'Data/*freqmax.npy'
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain, average energy, spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    geo_files = sorted(glob.glob(directory_path + '/' + geo_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmin_files = sorted(glob.glob(directory_path + '/' + freqmin_wildcard_pattern), key=lambda x: os.path.basename(x))
    freqmax_files = sorted(glob.glob(directory_path + '/' + freqmax_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get relative geometries and frequency ranges
    geos = []; freqrange = []
    for geo_file, freqmin_file, freqmax_file in zip(geo_files, freqmin_files, freqmax_files):
        geo_i = np.load(geo_file); freqmin_i = np.load(freqmin_file); freqmax_i = np.load(freqmax_file)
        ref_idx = np.where((geo_i == 0.))
        try:
            ref_idx = np.where((geo_i == 0.)); geo_i[ref_idx] = np.array([1e-10, 1e-10]) # need to mask out zeros for standardization
        except:
            for zero_idx in range(len(ref_idx[0])):
                geo_i[ref_idx[0][zero_idx], ref_idx[1][zero_idx]] = 1e-10
        geos.append(geo_i); freqrange.append(np.array([freqmin_i, freqmax_i]))
    X_geos_seismic.append(geos); X_freqrange_seismic.append(freqrange)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC_max = []; RC_mean = []; RC_std = []; RC_single_output = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_max.append(RC_max_i)
        RC_mean_i = np.load(mean_file); RC_mean.append(RC_mean_i)
        RC_std_i = np.load(std_file); RC_std.append(RC_std_i)
    RC_max = np.array(RC_max).reshape(-1,1); RC_mean = np.array(RC_mean).reshape(-1,1); RC_std = np.array(RC_std).reshape(-1,1)
    RC = np.zeros((RC_max.shape[0],3))
    RC[:,0] = RC_max[:,0]; RC[:,1] = RC_mean[:,0]; RC[:,2] = RC_std[:,0] # 1st column is max, 2nd column is mean, 3rd column is std
    y_RC_seismic.append(RC)
    #-----------------------------------------------------------------------------------------------------------------#
    print('Done with band: '+ str(band_idx+1))

Done with band: 10
CPU times: user 18.9 s, sys: 3min 22s, total: 3min 40s
Wall time: 9min 52s


---
Center pad geometry data

In [79]:
%%time
# Pad infrasound geometries
X_geometries_infrasound_padded = X_geos.copy()
for idx1 in range(len(X_geos)):
    for idx2 in range(len(X_geos[idx1])):
        # Center Pad
        padded_geometry = cardinal.pad_geometry(X_geos[idx1][idx2], max_stns=10)
        X_geometries_infrasound_padded[idx1][idx2] = padded_geometry
X_geometries_infrasound_padded = np.concatenate((X_geometries_infrasound_padded))

CPU times: user 2.43 s, sys: 66.8 ms, total: 2.49 s
Wall time: 2.48 s


In [82]:
%%time
# Save infrasound
save = False
if save == True:
    np.save('X_geometries_infrasound_padded.npy', X_geometries_infrasound_padded)
    np.save('X_freqranges_infrasound.npy', np.concatenate((X_freqrange)))
    np.save('y_RCs_infrasound.npy', np.concatenate((y_RC)))

CPU times: user 45.1 ms, sys: 22 ms, total: 67.1 ms
Wall time: 88.2 ms


In [62]:
%%time
# Pad seismic geometries
X_geometries_seismic_padded = X_geos_seismic.copy()
for idx1 in range(len(X_geos_seismic)):
    for idx2 in range(len(X_geos_seismic[idx1])):
        # Center Pad
        padded_geometry = cardinal.pad_geometry(X_geos_seismic[idx1][idx2], max_stns=10)
        X_geometries_seismic_padded[idx1][idx2] = padded_geometry
X_geometries_seismic_padded = np.concatenate((X_geometries_seismic_padded))

CPU times: user 1.09 s, sys: 37.8 ms, total: 1.13 s
Wall time: 1.12 s


In [76]:
%%time
# Save seismic
save = False
if save == True:
    np.save('X_geometries_seismic_padded.npy', X_geometries_seismic_padded)
    np.save('X_freqranges_seismic.npy', np.concatenate((X_freqrange_seismic)))
    np.save('y_RCs_seismic.npy', np.concatenate((y_RC_seismic)))

CPU times: user 24.9 ms, sys: 10.9 ms, total: 35.9 ms
Wall time: 41.1 ms


---
Load datasets and construct single RC metric (max, mean, std)

In [33]:
%%time
# Load infrasound
X_geos_infra = np.load('X_geometries_infrasound_padded.npy', allow_pickle=True)
X_freq_infra = np.load('X_freqranges_infrasound.npy', allow_pickle=True)
y_RCs_infra = np.load('y_RCs_infrasound.npy', allow_pickle=True)

CPU times: user 2.11 ms, sys: 25.9 ms, total: 28 ms
Wall time: 74.6 ms


In [115]:
%%time
# Construct RC metric for infrasound
RC_metric_infrasound = []
for RC_params in y_RCs_infra:
    RC_metric_tmp = (RC_params[1] + RC_params[2]) / RC_params[0]
    RC_metric_infrasound.append(RC_metric_tmp)
RC_metric_infrasound = np.array(RC_metric_infrasound)
RC_metric_infrasound = RC_metric_infrasound.reshape(RC_metric_infrasound.shape[0],1) # reshape to have 1 column

CPU times: user 96.8 ms, sys: 5.87 ms, total: 103 ms
Wall time: 101 ms


In [116]:
%%time
# Save infrasound RC metric
save = False
if save == True:
    np.save('RC_metric_infrasound.npy', RC_metric_infrasound)

CPU times: user 1.04 ms, sys: 5.21 ms, total: 6.26 ms
Wall time: 3.53 ms


In [34]:
%%time
# Load seismic
X_geos_seismic = np.load('X_geometries_seismic_padded.npy', allow_pickle=True)
X_freq_seismic = np.load('X_freqranges_seismic.npy', allow_pickle=True)
y_RCs_seismic = np.load('y_RCs_seismic.npy', allow_pickle=True)

CPU times: user 1.68 ms, sys: 19.8 ms, total: 21.5 ms
Wall time: 44.7 ms


In [118]:
%%time
# Construct RC metric for seismic
RC_metric_seismic = []
for RC_params in y_RCs_seismic:
    RC_metric_tmp = (RC_params[1] + RC_params[2]) / RC_params[0]
    RC_metric_seismic.append(RC_metric_tmp)
RC_metric_seismic = np.array(RC_metric_seismic)
RC_metric_seismic = RC_metric_seismic.reshape(RC_metric_seismic.shape[0],1) # reshape to have 1 column

CPU times: user 60.9 ms, sys: 3.54 ms, total: 64.5 ms
Wall time: 62.5 ms


In [119]:
%%time
# Save seismic RC metric
save = False
if save == True:
    np.save('RC_metric_seismic.npy', RC_metric_seismic)

CPU times: user 1.07 ms, sys: 3.49 ms, total: 4.56 ms
Wall time: 3.25 ms


---
Load seismic and infrasound RC metric

In [35]:
%%time
# Load infrasound
RC_metric_infrasound = np.load('RC_metric_infrasound.npy', allow_pickle=True)

CPU times: user 1.01 ms, sys: 2.6 ms, total: 3.62 ms
Wall time: 8.36 ms


In [36]:
%%time
# Load seismic
RC_metric_seismic = np.load('RC_metric_seismic.npy', allow_pickle=True)

CPU times: user 1.29 ms, sys: 3.91 ms, total: 5.2 ms
Wall time: 7.65 ms


In [37]:
print('Total training samples: '+str(np.concatenate((RC_metric_infrasound, RC_metric_seismic)).shape[0]))

Total training samples: 485086


## Create Training Datasets
---
kfold cross val

In [39]:
# Precompute splits
n_splits = 5; split = 0
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
#-----------------------------------------------------------------------------------------------------------------#
# Splitting based on indices
seismic_indices = np.arange(len(X_geos_seismic))
infrasound_indices = np.arange(len(X_geos_infra))
for fold, (seismic_train_idx, seismic_val_idx) in enumerate(kf.split(seismic_indices)):
    infrasound_train_idx, infrasound_val_idx = next(kf.split(infrasound_indices))  # Independent split
    # Split seismic and infrasound array geometries
    X_seismic_train, X_seismic_val = X_geos_seismic[seismic_train_idx], X_geos_seismic[seismic_val_idx]
    X_infrasound_train, X_infrasound_val = X_geos_infra[infrasound_train_idx], X_geos_infra[infrasound_val_idx]
    # Split seismic and infrasound frequency ranges
    F_seismic_train, F_seismic_val = X_freq_seismic[seismic_train_idx], X_freq_seismic[seismic_val_idx]
    F_infrasound_train, F_infrasound_val = X_freq_infra[infrasound_train_idx], X_freq_infra[infrasound_val_idx]
    F_train = np.concatenate([F_seismic_train, F_infrasound_train], axis=0) # concatenate frequency ranges since they are not scaled independently
    F_val = np.concatenate([F_seismic_val, F_infrasound_val], axis=0)
    # Split seismic and infrasound RC target
    y_seismic_train, y_seismic_val = RC_metric_seismic[seismic_train_idx], RC_metric_seismic[seismic_val_idx]
    y_infrasound_train, y_infrasound_val = RC_metric_infrasound[infrasound_train_idx], RC_metric_infrasound[infrasound_val_idx]
    #-----------------------------------------------------------------------------------------------------------------#
    # Scaling array geometries and targets independently - scaling combined frequency ranges
    X_seismic_train_scaled, X_seismic_val_scaled, X_seismic_scaler = cardinal.scale_padded_dataset(X_seismic_train, X_seismic_val)
    X_infrasound_train_scaled, X_infrasound_val_scaled, X_infrasound_scaler = cardinal.scale_padded_dataset(X_infrasound_train, X_infrasound_val)
    F_train_scaled, F_val_scaled, F_scaler = cardinal.scale_padded_dataset(F_train, F_val)
    y_seismic_train_scaled, y_seismic_val_scaled, y_seismic_scaler = cardinal.scale_padded_dataset(y_seismic_train, y_seismic_val)
    y_infrasound_train_scaled, y_infrasound_val_scaled, y_infrasound_scaler = cardinal.scale_padded_dataset(y_infrasound_train, y_infrasound_val)
    #-----------------------------------------------------------------------------------------------------------------#
    # Split scaled frequencies back into seismic and infrasound sets
    F_seismic_train_scaled, F_infrasound_train_scaled = np.split(F_train_scaled, [len(F_seismic_train)], axis=0)
    F_seismic_val_scaled, F_infrasound_val_scaled = np.split(F_val_scaled, [len(F_seismic_val)], axis=0)
    #-----------------------------------------------------------------------------------------------------------------#
    # OHE seismic and infrasound train/test input geometries AFTER SCALING! S
    # Seismic
    seismic_train_ohe = np.tile([1,0], (X_seismic_train_scaled.shape[0], X_seismic_train_scaled.shape[1], 1)) # seismic [1,0]
    X_seismic_train_scaled = np.concatenate((X_seismic_train_scaled, seismic_train_ohe), axis=2)
    seismic_val_ohe = np.tile([1,0], (X_seismic_val_scaled.shape[0], X_seismic_val_scaled.shape[1], 1)) # seismic [1,0]
    X_seismic_val_scaled = np.concatenate((X_seismic_val_scaled, seismic_val_ohe), axis=2)
    # Infrasound
    infra_train_ohe = np.tile([0,1], (X_infrasound_train_scaled.shape[0], X_infrasound_train_scaled.shape[1], 1)) # infrasound [0,1]
    X_infrasound_train_scaled = np.concatenate((X_infrasound_train_scaled, infra_train_ohe), axis=2)
    infra_val_ohe = np.tile([0,1], (X_infrasound_val_scaled.shape[0], X_infrasound_val_scaled.shape[1], 1)) # infrasound [0,1]
    X_infrasound_val_scaled = np.concatenate((X_infrasound_val_scaled, infra_val_ohe), axis=2)
    #-----------------------------------------------------------------------------------------------------------------#
    # Concatenate seismic & infrasound **after** independent scaling
    X_train_final = np.concatenate([X_seismic_train_scaled, X_infrasound_train_scaled], axis=0) # train synthetic arrays
    X_val_final = np.concatenate([X_seismic_val_scaled, X_infrasound_val_scaled], axis=0) # test synthetic arrays
    F_train_final = np.concatenate([F_seismic_train_scaled, F_infrasound_train_scaled], axis=0) # train freqs
    F_val_final = np.concatenate([F_seismic_val_scaled, F_infrasound_val_scaled], axis=0) # test freqs
    y_train_final = np.concatenate([y_seismic_train_scaled, y_infrasound_train_scaled], axis=0) # train RCs
    y_val_final = np.concatenate([y_seismic_val_scaled, y_infrasound_val_scaled], axis=0) # test RCs
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SHUFFLE TRAIN AGAIN (KFold is barely shuffling the data)
    random_train_aug_segments_idx = random.sample(range(X_train_final.shape[0]), X_train_final.shape[0])
    X_train_final = X_train_final[random_train_aug_segments_idx,:,:]
    F_train_final = F_train_final[random_train_aug_segments_idx,:]
    y_train_final = y_train_final[random_train_aug_segments_idx,:]
    #-----------------------------------------------------------------------------------------------------------------#
    # Shuffling test
    random_test_aug_segments_idx = random.sample(range(X_val_final.shape[0]), X_val_final.shape[0])
    X_val_final = X_val_final[random_test_aug_segments_idx,:,:]
    F_val_final = F_val_final[random_test_aug_segments_idx,:]
    y_val_final = y_val_final[random_test_aug_segments_idx,:]
    #-----------------------------------------------------------------------------------------------------------------#
    # Save train/test sets and scalers    
    np.save('Training_Data/Split_'+str(split+1)+'/X_train_final.npy', X_train_final); np.save('Training_Data/Split_'+str(split+1)+'/X_val_final.npy', X_val_final)
    np.save('Training_Data/Split_'+str(split+1)+'/F_train_final.npy', F_train_final); np.save('Training_Data/Split_'+str(split+1)+'/F_val_final.npy', F_val_final)
    np.save('Training_Data/Split_'+str(split+1)+'/y_train_final.npy', y_train_final); np.save('Training_Data/Split_'+str(split+1)+'/y_val_final.npy', y_val_final)
    with open('Training_Data/Split_'+str(split+1)+'/X_seismic_scaler.pkl', 'wb') as f: # seismic array scaler
        pickle.dump(X_seismic_scaler, f)
    with open('Training_Data/Split_'+str(split+1)+'/X_infrasound_scaler.pkl', 'wb') as f: # infrasound array scaler
        pickle.dump(X_infrasound_scaler, f)
    with open('Training_Data/Split_'+str(split+1)+'/F_scaler.pkl', 'wb') as f: # frequency scaler
        pickle.dump(F_scaler, f)        
    with open('Training_Data/Split_'+str(split+1)+'/y_seismic_scaler.pkl', 'wb') as f: # seismic RC scaler
        pickle.dump(y_seismic_scaler, f)         
    with open('Training_Data/Split_'+str(split+1)+'/y_infrasound_scaler.pkl', 'wb') as f: # infrasound RC scaler
        pickle.dump(y_infrasound_scaler, f)       
    #-----------------------------------------------------------------------------------------------------------------#
    print('Done saving quantile transformer scaled data')
    split += 1

Done saving quantile transformer scaled data
Done saving quantile transformer scaled data
Done saving quantile transformer scaled data
Done saving quantile transformer scaled data
Done saving quantile transformer scaled data


## TESTING: Optimize Weighted RC Sum (not included in paper)
---
#### Each frequency band will have own optimized weights

In [52]:
from scipy.optimize import minimize

# Function to calculate the weighted sum of mean and standard deviation
def calculate_weighted_sum(mean, std, alpha, beta):
    return alpha * mean + beta * std

# MSE objective function
def objective_mse(weights, means, stds, target_score):
    alpha, beta = weights
    total_error = 0
    for mean_tmp, std_tmp in zip(means, stds):
        weighted_sum = calculate_weighted_sum(mean_tmp, std_tmp, alpha, beta)
        total_error += (weighted_sum - target_score) ** 2
    mse = total_error / len(means)
    return mse

# Example function to find optimal weights
def find_optimal_weights(means, stds, target_score):
    initial_weights = [0.5, 0.5]
    result = minimize(objective_mse, initial_weights, args=(means, stds, target_score), bounds=[(0, 1), (0, 1)])
    return result.x

In [35]:
%%time
alphas = []; betas = []
for band_idx in range(len(f_bands)):
    # Get filepaths for subarray geometries, freq ranges, and RC (max, mean, std)
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain, average energy, spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC_max = []; RC_mean = []; RC_std = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_max.append(RC_max_i)
        RC_mean_i = np.load(mean_file); RC_mean.append(RC_mean_i)
        RC_std_i = np.load(std_file); RC_std.append(RC_std_i)
    RC_max = np.array(RC_max).reshape(-1,1); RC_mean = np.array(RC_mean).reshape(-1,1); RC_std = np.array(RC_std).reshape(-1,1)
    #-----------------------------------------------------------------------------------------------------------------#
    # Begin optimization of weighted sum
    target_score = 1e-3
    optimal_weights = find_optimal_weights(RC_mean, RC_std, target_score)
    alpha, beta = optimal_weights
    print(f"Optimal weights: alpha = {alpha}, beta = {beta}")
    alphas.append(alpha); betas.append(beta)
    print('Done with band: '+ str(band_idx+1))

Optimal weights: alpha = 0.0009961107040981551, beta = 0.0013070274101782556
Done with band: 1
Optimal weights: alpha = 0.0009034819441055308, beta = 0.0019409549920416937
Done with band: 2
Optimal weights: alpha = 0.000734405770304254, beta = 0.0029625543486175247
Done with band: 3
Optimal weights: alpha = 0.0007000982897625023, beta = 0.003945092644695477
Done with band: 4
Optimal weights: alpha = 6.405321303251974e-05, beta = 0.006971789510488461
Done with band: 5
Optimal weights: alpha = 0.0012210531353190625, beta = 0.004764232977367161
Done with band: 6
Optimal weights: alpha = 0.0026500890490058185, beta = 0.0023772174502841924
Done with band: 7
Optimal weights: alpha = 0.0027245868971190914, beta = 0.0030003222304878473
Done with band: 8
Optimal weights: alpha = 0.0018751298337705794, beta = 0.005446016132914118
Done with band: 9
Optimal weights: alpha = 0.0014942874061101854, beta = 0.006858298859991145
Done with band: 10
CPU times: user 13min 5s, sys: 7min 8s, total: 20min 14

In [54]:
%%time
# Trying with higher target score
alphas = []; betas = []
for band_idx in range(len(f_bands)):
    # Get filepaths for subarray geometries, freq ranges, and RC (max, mean, std)
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy' # array gain, average energy, spread of energy
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC_max = []; RC_mean = []; RC_std = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_max.append(RC_max_i)
        RC_mean_i = np.load(mean_file); RC_mean.append(RC_mean_i)
        RC_std_i = np.load(std_file); RC_std.append(RC_std_i)
    RC_max = np.array(RC_max).reshape(-1,1); RC_mean = np.array(RC_mean).reshape(-1,1); RC_std = np.array(RC_std).reshape(-1,1)
    #-----------------------------------------------------------------------------------------------------------------#
    # Begin optimization of weighted sum
    target_score = 1e-1 # 0.1
    optimal_weights = find_optimal_weights(RC_mean, RC_std, target_score)
    alpha, beta = optimal_weights
    print(f"Optimal weights: alpha = {alpha}, beta = {beta}")
    alphas.append(alpha); betas.append(beta)
    print('Done with band: '+ str(band_idx+1))

Optimal weights: alpha = 0.09961291572380782, beta = 0.13069259375804154
Done with band: 1
Optimal weights: alpha = 0.09035289761866422, beta = 0.19407738009665335
Done with band: 2
Optimal weights: alpha = 0.0734419313863193, beta = 0.2962534205784935
Done with band: 3
Optimal weights: alpha = 0.07001142512233823, beta = 0.39450630406033455
Done with band: 4
Optimal weights: alpha = 0.058411265237474926, beta = 0.5576257945809074
Done with band: 5
Optimal weights: alpha = 0.12210996836098166, beta = 0.476411266010574
Done with band: 6
Optimal weights: alpha = 0.26500924100645123, beta = 0.2377162460899642
Done with band: 7
Optimal weights: alpha = 0.27245999690748746, beta = 0.3000275752142239
Done with band: 8
Optimal weights: alpha = 0.18751524807963385, beta = 0.5445931415463134
Done with band: 9
Optimal weights: alpha = 0.1494296374215301, beta = 0.6858099390724042
Done with band: 10
CPU times: user 9min 46s, sys: 7min 32s, total: 17min 18s
Wall time: 1h 17min 21s


In [55]:
%%time
save = False
if save == True:
    np.save('Weights/alphas_0_1.npy', np.array(alphas))
    np.save('Weights/betas_0_1.npy', np.array(betas))

CPU times: user 1.21 ms, sys: 2.75 ms, total: 3.96 ms
Wall time: 6.85 ms


---
#### Now sort ARF's using weighted sum and max - RC = (alpha * mean + beta * std) / max

In [43]:
%%time
alphas = np.load('Weights/alphas.npy')
betas = np.load('Weights/betas.npy')
for band_idx in range(len(f_bands)):
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, mean, std) and weights
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy'
    alpha = alphas[band_idx]; beta = betas[band_idx]
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC = []; subarray = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_mean_i = np.load(mean_file); RC_std_i = np.load(std_file); 
        RC_tmp = (alpha * RC_mean_i + beta * RC_std_i) / RC_max_i # calculate RC
        RC.append(RC_tmp)
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 1
Worst configuration: Synthetic_Array_Revised_Max/Band_1/Plots/7076_S3_SB_SE_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_1/Plots/66411_SE_S1_SD_SC_S2_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_1/Plots/27667_S5_SB_SA_SE_S4_SD_SC_config.png
Start band 2
Worst configuration: Synthetic_Array_Revised_Max/Band_2/Plots/132021_SE_SB_S3_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_2/Plots/122054_SE_SD_S3_S4_SB_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_2/Plots/123462_SA_S1_SD_SC_SB_SE_S5_config.png
Start band 3
Worst configuration: Synthetic_Array_Revised_Max/Band_3/Plots/179937_SE_SA_S3_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_3/Plots/170154_SE_SA_SB_S2_S3_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_3/Plots/212867_S3_SD_SE_SB_SC_SA_S4_config.png
Start band 4
Worst configuration: Synthetic_Array_Revised_Max/Band_4/Plots/91659_S5_SB_SA_config.png
Medium co

In [57]:
%%time
alphas = np.load('Weights/alphas_0_1.npy')
betas = np.load('Weights/betas_0_1.npy')
for band_idx in range(len(f_bands)):
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, mean, std) and weights
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy'
    alpha = alphas[band_idx]; beta = betas[band_idx]
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC = []; subarray = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_mean_i = np.load(mean_file); RC_std_i = np.load(std_file); 
        RC_tmp = (alpha * RC_mean_i + beta * RC_std_i) / RC_max_i # calculate RC
        RC.append(RC_tmp)
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 1
Worst configuration: Synthetic_Array_Revised_Max/Band_1/Plots/7076_S3_SB_SE_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_1/Plots/195736_SD_SE_SA_S2_SB_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_1/Plots/27667_S5_SB_SA_SE_S4_SD_SC_config.png
Start band 2
Worst configuration: Synthetic_Array_Revised_Max/Band_2/Plots/132021_SE_SB_S3_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_2/Plots/233519_SA_S4_SB_S1_S2_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_2/Plots/123462_SA_S1_SD_SC_SB_SE_S5_config.png
Start band 3
Worst configuration: Synthetic_Array_Revised_Max/Band_3/Plots/179937_SE_SA_S3_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_3/Plots/170154_SE_SA_SB_S2_S3_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_3/Plots/212867_S3_SD_SE_SB_SC_SA_S4_config.png
Start band 4
Worst configuration: Synthetic_Array_Revised_Max/Band_4/Plots/91659_S5_SB_SA_config.png
Medium c

---
#### Without weights this time

In [58]:
%%time
for band_idx in range(len(f_bands)):
    print('Start band ' +str(band_idx+1))
    # Get filepaths for RC (max, mean, std) and weights
    directory_path = 'Synthetic_Arrays/Band_'+str(band_idx+1)
    RC_max_wildcard_pattern = 'Data/*_max.npy'; RC_mean_wildcard_pattern = 'Data/*mean.npy'; RC_std_wildcard_pattern = 'Data/*std.npy'
    #-----------------------------------------------------------------------------------------------------------------#
    # NEED TO SORT ALL FILES THE SAME WAY (doesn't matter how they're sorted - just that it's the same)
    RC_max_files = sorted(glob.glob(directory_path + '/' + RC_max_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_mean_files = sorted(glob.glob(directory_path + '/' + RC_mean_wildcard_pattern), key=lambda x: os.path.basename(x))
    RC_std_files = sorted(glob.glob(directory_path + '/' + RC_std_wildcard_pattern), key=lambda x: os.path.basename(x))
    #-----------------------------------------------------------------------------------------------------------------#
    # Get RC (max, mean, std)
    RC = []; subarray = []
    for max_file, mean_file, std_file in zip(RC_max_files, RC_mean_files, RC_std_files):
        RC_max_i = np.load(max_file); RC_mean_i = np.load(mean_file); RC_std_i = np.load(std_file); 
        RC_tmp = (RC_mean_i + RC_std_i) / RC_max_i # calculate RC
        RC.append(RC_tmp)
        subarray_i = std_file.split('/')[-1][:-8]; subarray.append(subarray_i)
    RC = np.array(RC); subarray = np.array(subarray)
    #-----------------------------------------------------------------------------------------------------------------#
    # Get images
    RC_sorted = np.argsort(RC)[::-1] # indices from max to min RC
    imgs = []
    for subarray_i in subarray[RC_sorted]:
        imgs.append(directory_path + '/Plots/'+subarray_i+'_config.png')
    print('Worst configuration: ' +imgs[0])
    print('Medium configuration: '+imgs[len(imgs) // 2])
    print('Best configuration: '+imgs[-1])

Start band 1
Worst configuration: Synthetic_Array_Revised_Max/Band_1/Plots/108027_S3_S1_SC_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_1/Plots/78415_SC_SD_S3_SA_SE_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_1/Plots/69412_SD_S3_SE_SB_SC_S4_SA_config.png
Start band 2
Worst configuration: Synthetic_Array_Revised_Max/Band_2/Plots/184037_S3_S4_SB_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_2/Plots/206437_SC_S5_SA_S2_S3_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_2/Plots/165390_S3_S5_SA_SB_SC_SE_SD_config.png
Start band 3
Worst configuration: Synthetic_Array_Revised_Max/Band_3/Plots/204030_S5_S1_SB_config.png
Medium configuration: Synthetic_Array_Revised_Max/Band_3/Plots/144569_S3_S2_S5_SA_S4_S1_config.png
Best configuration: Synthetic_Array_Revised_Max/Band_3/Plots/212867_S3_SD_SE_SB_SC_SA_S4_config.png
Start band 4
Worst configuration: Synthetic_Array_Revised_Max/Band_4/Plots/142913_SE_S2_S5_config.png
Med